In [ ]:
# ── Hücre 1: Drive Bağlama + Repo Klonlama ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/TDL-ADD/models/A2_Mamba/checkpoints_A2_Mamba', exist_ok=True)
os.makedirs('/content/drive/MyDrive/TDL-ADD/scores/A2_Mamba', exist_ok=True)
os.makedirs('/content/drive/MyDrive/TDL-ADD/wheels', exist_ok=True)

if not os.path.exists('/content/TDL-ADD'):
    !git clone https://github.com/CenkAydin/TDL-ADD /content/TDL-ADD
else:
    print('Repo zaten mevcut, güncelleniyor...')
    !git -C /content/TDL-ADD pull

%cd /content/TDL-ADD
print('Çalışma dizini:', os.getcwd())

# Idle timeout engelleyici (90 dk sonra session ölmesin)
from IPython.display import display, Javascript
display(Javascript("""
function keepAlive() {
    console.log('keep-alive ping');
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(keepAlive, 55000);
"""))
print('Keep-alive aktif.')

In [ ]:
# ── Hücre 2: Akıllı Mamba Kurulumu (Drive Wheel Cache) ────────────────────────
import os, glob, shutil

os.environ['CUDA_HOME']                 = '/usr/local/cuda'
os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
os.environ['MAMBA_FORCE_BUILD']         = 'TRUE'
os.environ['MAX_JOBS']                  = '4'

!nvcc --version
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)"
!pip install packaging ninja --quiet

WHEEL_DIR = '/content/drive/MyDrive/TDL-ADD/wheels'
os.makedirs(WHEEL_DIR, exist_ok=True)

# Drive'da önceden kaydedilmiş wheel var mı kontrol et
cached_causal = glob.glob(f"{WHEEL_DIR}/causal_conv1d*.whl")
cached_mamba  = glob.glob(f"{WHEEL_DIR}/mamba_ssm*.whl")

if cached_causal and cached_mamba:
    print(f"\nDrive'da hazır wheel bulundu! Kuruluyor...")
    print(f"  causal-conv1d: {os.path.basename(cached_causal[0])}")
    print(f"  mamba-ssm:     {os.path.basename(cached_mamba[0])}")
    causal_whl = cached_causal[0]
    mamba_whl  = cached_mamba[0]
    !pip install {causal_whl} --quiet
    !pip install {mamba_whl}  --quiet
else:
    print("\nDrive'da wheel yok. Mamba sıfırdan derleniyor (~45-50 dk)...")
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm     --no-build-isolation

    # Wheel'ları Drive'a Python ile kaydet (shutil kullanıyoruz, shell {} çakışması yok)
    print("\nDerleme tamamlandı. Wheel'lar Drive'a kaydediliyor...")
    all_cached = glob.glob('/root/.cache/pip/wheels/**/*.whl', recursive=True)
    targets = [w for w in all_cached
               if 'causal_conv1d' in os.path.basename(w) or 'mamba_ssm' in os.path.basename(w)]
    if not targets:
        print("UYARI: Pip cache'de wheel bulunamadi. Dizin kontrol ediliyor...")
        !find /root/.cache/pip/wheels/ -name "*.whl" | grep -E "causal|mamba"
    else:
        for w in targets:
            dest = os.path.join(WHEEL_DIR, os.path.basename(w))
            shutil.copy2(w, dest)
            print(f"  Kaydedildi: {os.path.basename(w)}")
        print(f"Toplam {len(targets)} wheel Drive'a kaydedildi.")

# Kurulumu doğrula
!python -c "from mamba_ssm import Mamba; print('mamba-ssm OK')"
!pip install transformers tqdm pytorch-model-summary --quiet

In [ ]:
# ── Hücre 3: Veri + Preprocess (Drive-first, akıllı extraction) ───────────────
import os, glob, subprocess, shutil

BASE_DIR       = '/content/asv2019PS'
DATA_ROOT      = '/content/asv2019PS/database'
FEATURE_ROOT   = '/content/asv2019PS/preprocess_A1_WavLM_Large'
ARCHIVE_PATH   = '/content/asv2019ps_archive.zip'
RAW_DIR        = '/content/asv2019PS_raw'
DRIVE_FEATURES = '/content/drive/MyDrive/TDL-ADD/preprocess_A1_WavLM_Large'

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# ─── ADIM 0: Drive'da feature'lar var mı? ──
if os.path.exists(os.path.join(DRIVE_FEATURES, 'train', 'wavlm-large')):
    print("Drive'da feature'lar bulundu! /content/'e kopyalanıyor (~20-40 dk)...")
    !cp -r {DRIVE_FEATURES} /content/asv2019PS/
    print("Kopyalama tamamlandi. Adim 1-4 atlaniyor.")
else:
    print("Drive'da feature yok. Indirme + preprocess basliyor...\n")

    # ─── ADIM 1: Zenodo arşivini indir ──────────────────────────────────────────
    archive_ok = False
    if os.path.exists(ARCHIVE_PATH):
        size_gb = os.path.getsize(ARCHIVE_PATH) / 1e9
        print(f"Arsiv mevcut: {size_gb:.2f} GB")
        if size_gb >= 20.0:
            archive_ok = True
            print("Boyut yeterli, indirme atlandi.")
        else:
            print(f"Arsiv kucuk ({size_gb:.2f} GB < 20 GB), wget --continue ile devam ediliyor...")

    if not archive_ok:
        print("Zenodo arsivi indiriliyor (~25 GB)...")
        !wget --continue --show-progress \
            'https://zenodo.org/api/records/5766198/files-archive' \
            -O {ARCHIVE_PATH}
        final_size = os.path.getsize(ARCHIVE_PATH) / 1e9
        print(f"Indirme tamamlandi: {final_size:.2f} GB")
        if final_size < 20.0:
            raise RuntimeError(f"HATA: Arsiv beklenden kucuk ({final_size:.2f} GB). Indirme eksik!")

    # ─── ADIM 2: Zip'i aç ───────────────────────────────────────────────────────
    if not os.path.exists(RAW_DIR) or not glob.glob(f'{RAW_DIR}/*.tar.gz'):
        print("\nZip aciliyor...")
        !unzip -q {ARCHIVE_PATH} -d {RAW_DIR}
    else:
        print("Zip zaten acilmis.")

    print("\nZip icerigi:")
    !ls {RAW_DIR}

    # ─── ADIM 3: Her tar.gz'yi akıllıca çıkart ──────────────────────────────────
    if not os.path.exists(os.path.join(DATA_ROOT, 'train')):
        tar_files = sorted(glob.glob(f'{RAW_DIR}/*.tar.gz'))
        print(f"\n{len(tar_files)} adet tar.gz bulundu.")

        for f in tar_files:
            fname = os.path.basename(f)
            result = subprocess.run(['tar', '-tzf', f], capture_output=True, text=True)
            top_dirs = set(line.split('/')[0] for line in result.stdout.strip().split('\n')[:20] if line)
            print(f"  {fname} → ust klasorler: {top_dirs}")

            if 'database' in top_dirs:
                extract_target = BASE_DIR
            else:
                extract_target = DATA_ROOT

            print(f"    Cikariliyor → {extract_target}/")
            subprocess.run(['tar', '-xzf', f, '-C', extract_target], check=True)

        print("\nDatabase icerigi (train, dev, eval, protocols gorunmeli):")
        !ls {DATA_ROOT}
    else:
        print("Tar.gz dosyalari zaten cikarilmis.")
        !ls {DATA_ROOT}

    # ─── ADIM 4: WavLM-Large preprocess ────────────────────────────────────────
    if not os.path.exists(os.path.join(FEATURE_ROOT, 'train', 'wavlm-large')):
        print("\nOzellik cikarma basliyor (~3-4 saat)...")
        !python /content/TDL-ADD/preprocess.py \
            --database_dir {DATA_ROOT} \
            --protocol_dir /content/TDL-ADD/label \
            --output_dir   {FEATURE_ROOT}
    else:
        print("Ozellikler zaten cikarilmis.")

    # ─── ADIM 5: Feature'ları Drive'a yedekle ──────────────────────────────────
    if os.path.exists(os.path.join(FEATURE_ROOT, 'train', 'wavlm-large')):
        if not os.path.exists(os.path.join(DRIVE_FEATURES, 'train', 'wavlm-large')):
            print("\nFeature'lar Drive'a yedekleniyor (~60 GB, ~30-40 dk)...")
            !cp -r {FEATURE_ROOT} /content/drive/MyDrive/TDL-ADD/
            print("Drive yedekleme tamamlandi!")
        else:
            print("Drive yedeklemesi zaten mevcut.")
    else:
        print("UYARI: Preprocess basarisiz! Feature klasoru bulunamadi.")

print("\nAdim 3 ozeti — Feature root kontrolu:")
!ls {FEATURE_ROOT}

In [ ]:
# ── Hücre 4: Eğitimi Başlat ───────────────────────────────────────────────────
!python /content/TDL-ADD/main_train.py \
    -m  TDL_Mamba \
    -f  /content/asv2019PS/preprocess_A1_WavLM_Large \
    -d  /content/asv2019PS/database \
    -o  /content/drive/MyDrive/TDL-ADD/models/A2_Mamba \
    --ckpt_subdir checkpoints_A2_Mamba \
    --num_epochs  200 \
    --batch_size  24 \
    --lr          0.0001 \
    --lam         0.1 \
    --num_workers 2 \
    --base_loss   bce \
    --gpu         0